# Embarrassingly parallel R on Snowflake (SPCS + doSnowflake)

Forecasting here is only an **illustration**; the same patterns apply to HPO grids, back-tests, Monte Carlo, etc.

**Prerequisites:** `workspace_parallel_spcs_setup.ipynb` once.

**Config:** `parallel_lab` + optional `context` in `snowflaker_parallel_spcs_config.yaml` (via `parallel_lab_config.py`).
**Dev default:** YAML points at **MR Price** warehouse / pool / DB already used by internal doSnowflake benchmarks; swap to generic names before publishing.
**Run gate:** §1b — keep `RUN_PARALLEL_DEMO` false in git; enable via scratch cell when running SPCS cells.
**Monitoring:** driver cells block — use `workspace_parallel_spcs_monitor.ipynb` or Streamlit in another tab.


## 1. Environment


In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(
    config="snowflaker_parallel_spcs_config.yaml",
    packages=["snowflakeR", "RSnowflake"]
)
import parallel_lab_config as plc
plc.ensure_notebook_path_on_sys_path()
LAB = plc.apply_parallel_lab_environment(plc.load_parallel_lab_dict())
print("parallel_lab:", LAB["database"], "| pool:", LAB.get("compute_pool") or "(unset)")


## 1b. Run gate (Workspace-friendly)

In Snowflake Workspace, **editing** cells creates version drift against git. Keep **`RUN_PARALLEL_DEMO = False`** in the next cell in the committed notebook so accidental **Run all** does not launch SPCS work.

**When you intend to run** `registerDoSnowflake` / registry cells below, use a **scratch cell** (not committed) to enable, then re-run the distributed cells:

```python
import os
os.environ["PARALLEL_LAB_RUN_DEMO"] = "1"
print("SPCS / doSnowflake cells enabled for this kernel session.")
```

Or set **`RUN_PARALLEL_DEMO = True`** in the next cell and execute it (that edit will be local until you revert).

R cells check **`Sys.getenv("PARALLEL_LAB_RUN_DEMO") == "1"`** before launching work.


In [ ]:
import os

# Keep False in git: prevents accidental SPCS launches from "Run all".
RUN_PARALLEL_DEMO = False

os.environ["PARALLEL_LAB_RUN_DEMO"] = "1" if RUN_PARALLEL_DEMO else "0"
print("PARALLEL_LAB_RUN_DEMO =", os.environ["PARALLEL_LAB_RUN_DEMO"])


## 2. Connect (R)


In [ ]:
%%R
library(snowflakeR)
library(dplyr)
library(dbplyr)
library(RSnowflake)
ep_db <- Sys.getenv("PARALLEL_LAB_DATABASE")
sch_src <- Sys.getenv("PARALLEL_LAB_SCHEMA_SOURCE_DATA")
wh <- Sys.getenv("PARALLEL_LAB_WAREHOUSE")
conn <- sfr_connect()
if (nzchar(wh)) {
  sfr_execute(conn, paste("USE WAREHOUSE", wh))
}
conn <- sfr_use(conn, database = ep_db, schema = sch_src)
cat("Using", ep_db, ".", sch_src, "\n", sep = "")


## 3. Static work allocation (`doSnowflake` tasks mode)

Runtime values come from **`parallel_lab`** in YAML (`PARALLEL_LAB_*` env vars).

The next cell is **gated** by **`PARALLEL_LAB_RUN_DEMO`** (§1b). Unless you enable it, it only prints pool / image / stage and skips `registerDoSnowflake`.


In [ ]:
%%R
pool  <- Sys.getenv("PARALLEL_LAB_COMPUTE_POOL")
img   <- Sys.getenv("PARALLEL_LAB_IMAGE_URI")
stage <- Sys.getenv("PARALLEL_LAB_STAGE_AT")
run   <- identical(Sys.getenv("PARALLEL_LAB_RUN_DEMO"), "1")
cat("doSnowflake tasks would use:\n  compute_pool =", pool,
    "\n  image_uri =", img, "\n  stage =", stage,
    "\n  PARALLEL_LAB_RUN_DEMO =", Sys.getenv("PARALLEL_LAB_RUN_DEMO"), "\n", sep = "")

if (!run) {
  cat("Skipping registerDoSnowflake (tasks). Enable via §1b or scratch cell.\n")
} else {
  library(foreach)
  registerDoSnowflake(
    conn,
    mode           = "tasks",
    compute_pool   = pool,
    image_uri      = img,
    chunks_per_job = 4L
  )
  # out <- foreach(i = seq_len(n_units)) %dopar% { ... }
}


## 4. Elastic queue (`doSnowflake` queue mode)

Same **run gate** as §3. Prints queue FQN; launches only when **`PARALLEL_LAB_RUN_DEMO`** is `"1"`.


In [ ]:
%%R
run <- identical(Sys.getenv("PARALLEL_LAB_RUN_DEMO"), "1")
cat("Queue FQN:", Sys.getenv("PARALLEL_LAB_QUEUE_FQN"),
    " | PARALLEL_LAB_RUN_DEMO =", Sys.getenv("PARALLEL_LAB_RUN_DEMO"), "\n", sep = "")

if (!run) {
  cat("Skipping registerDoSnowflake (queue). Enable via §1b or scratch cell.\n")
} else {
  library(foreach)
  registerDoSnowflake(
    conn,
    mode          = "queue",
    worker_type   = "ephemeral",
    compute_pool  = Sys.getenv("PARALLEL_LAB_COMPUTE_POOL"),
    image_uri     = Sys.getenv("PARALLEL_LAB_IMAGE_URI"),
    n_workers     = 4L
  )
}


## 5. Model Registry & batch inference

Registry schema: **`Sys.getenv("PARALLEL_LAB_SCHEMA_MODELS")`** under **`Sys.getenv("PARALLEL_LAB_DATABASE")`**.

Bundled partitions for batch inference — see `PARALLEL_SPCS_DEMO.md`.


In [ ]:
%%R
run <- identical(Sys.getenv("PARALLEL_LAB_RUN_DEMO"), "1")
ep_models <- Sys.getenv("PARALLEL_LAB_SCHEMA_MODELS")
cat("Model registry target:", ep_db, ".", ep_models,
    " | PARALLEL_LAB_RUN_DEMO =", Sys.getenv("PARALLEL_LAB_RUN_DEMO"), "\n", sep = "")

if (!run) {
  cat("Skipping sfr_model_registry / sfr_log_model. Enable via §1b or scratch cell.\n")
} else {
  reg <- sfr_model_registry(conn, database = ep_db, schema = ep_models)
  # mv <- sfr_log_model(reg, ...)
}


## Summary

- One YAML file for names + SPCS parameters.
- Driver blocks — monitor elsewhere.
